In [7]:
import sys
import math

def calc_dist(coord1, coord2):
    x1, y1, z1 = coord1
    x2, y2, z2 = coord2
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

def generate_cst(pdb_file, known_start, known_end, output, fasta_offset=281):
    """
    known_start, known_end: resíduos flanqueadores com estrutura conhecida.
    O loop desconhecido está entre eles.
    Gera constraints entre as âncoras e seus vizinhos imediatos (dentro da região conhecida).
    """
    coords = {}
    with open(pdb_file) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                res_num = int(line[22:26].strip())
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords[res_num] = (x, y, z)

    # Vizinhos das âncoras dentro da região conhecida
    anchor_start_neighbor = known_start - 1  # resíduo antes do start
    anchor_end_neighbor   = known_end + 1    # resíduo depois do end

    for res in [known_start, known_end, anchor_start_neighbor, anchor_end_neighbor]:
        if res not in coords:
            raise ValueError(f"Resíduo {res} não encontrado no PDB.")

    d_start = calc_dist(coords[known_start], coords[anchor_start_neighbor])
    d_end   = calc_dist(coords[known_end],   coords[anchor_end_neighbor])
    d_span  = calc_dist(coords[known_start], coords[known_end])

    def to_rosetta(pdb_num):
        return pdb_num - fasta_offset + 1

    with open(output, "w") as out:
        out.write(f"AtomPair CA {to_rosetta(known_start)} CA {to_rosetta(known_start-1)} HARMONIC {d_start:.3f} 0.5\n")
        out.write(f"AtomPair CA {to_rosetta(known_end)} CA {to_rosetta(known_end+1)} HARMONIC {d_end:.3f} 0.5\n")
        out.write(f"AtomPair CA {to_rosetta(known_start)} CA {to_rosetta(known_end)} HARMONIC {d_span:.3f} 1.0\n")

    print(f"Gerado: {output}")
    print(f"  Âncoras: {known_start} e {known_end} | Loop: {known_start+1}-{known_end-1} ({known_end - known_start - 1} resíduos)")

In [8]:
# Constrain UCR1 (166-187)
generate_cst("../4wzi_ucr2.pdb", known_start=282, known_end=333, output="./constraints.cst")

Gerado: ./constraints.cst
  Âncoras: 282 e 333 | Loop: 283-332 (50 resíduos)


In [1]:
best_ids = []

with open("results.txt") as f:
    for line in f:
        parts = line.strip().split()  # or .split(',') for CSV
        if len(parts) > 1:
            best_ids.append(parts[1])

In [2]:
best_ids

['S_00000060',
 'S_00000056',
 'S_00000023',
 'S_00000069',
 'S_00000114',
 'S_00000071',
 'S_00000109',
 'S_00000083',
 'S_00000164',
 'S_00000160']

In [3]:
def renumber_pdb(input_pdb, output_pdb, start=166):
    with open(input_pdb, 'r') as f:
        lines = f.readlines()
    
    current_resnum = None
    new_resnum = start - 1
    
    with open(output_pdb, 'w') as out:
        for line in lines:
            if line.startswith(('ATOM', 'HETATM')):
                res = line[22:26].strip()
                if res != current_resnum:
                    current_resnum = res
                    new_resnum += 1
                line = line[:22] + f'{new_resnum:4d}' + line[26:]
            out.write(line)


In [6]:
import os
import glob

abinitio_dir = '/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/'

pdb_files = sorted(glob.glob(os.path.join(abinitio_dir, 'S*.pdb')))

# ordenar arquivos conforme a lista
pdb_files_sorted = sorted(
    pdb_files,
    key=lambda x: best_ids.index(os.path.basename(x).split('.')[0])
)

for i, file in enumerate(pdb_files_sorted):
    print(file, i+1)
    renumber_pdb(
        input_pdb=file,
        output_pdb=abinitio_dir + f"rosetta{i+1}.pdb",
        start=281
    )

/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000060.pdb 1
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000056.pdb 2
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000023.pdb 3
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000069.pdb 4
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000114.pdb 5
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000071.pdb 6
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000109.pdb 7
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000083.pdb 8
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000164.pdb 9
/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/protein/again/abinitiolr2/S_00000